In [4]:
import os
import pandas as pd
gcd_exp = []
lar_list = [0.1, 0.5, 1.0]
dataset_cols = [
    '20NG', 'THUCNews', 'Yahoo', 'TREC', 'BANK',
    'S.O.', 'CLINC', 'HWU', 'ECDT', 'M-CID', 'DBPedia', 'X-Topic'
]

In [5]:
gcd_method_list = ['DAL', 'GeoID', 'SDC', 'DPN', 'TAN', 'LOOP', 'ALUP']
gcd_exp = []
for i in range(14, 32):
    df = pd.read_csv(f"table_or_csv/gcd/table_s{i}.csv")
    df['LAR'] = lar_list[(i - 2) % 3]
    gcd_exp.append(df)
gcd_exp = pd.concat(gcd_exp, ignore_index=True)
gcd_exp[dataset_cols] = gcd_exp[dataset_cols].apply(
    pd.to_numeric, errors="coerce"
)
gcd_exp = gcd_exp[gcd_exp['Method'].isin(gcd_method_list)]
gcd_exp["Method"] = pd.Categorical(
    gcd_exp["Method"],
    categories=gcd_method_list,
    ordered=True
)
# 1. 计算 Avg.
gcd_exp["Avg."] = gcd_exp[dataset_cols].mean(axis=1)


In [6]:
import numpy as np

def highlight_best_second(df, dataset_cols):
    df = df.copy()

    for metric in df.index.get_level_values("Metric").unique():
        sub_idx = df.index.get_level_values("Metric") == metric
        sub_df = df.loc[sub_idx, dataset_cols]

        for col in dataset_cols:
            values = sub_df[col]

            # 排序，忽略 NaN
            sorted_vals = values.dropna().sort_values(ascending=False)

            if len(sorted_vals) == 0:
                continue

            best = sorted_vals.iloc[0]
            second = sorted_vals.iloc[1] if len(sorted_vals) > 1 else None

            for idx, val in values.items():
                if pd.isna(val):
                    continue
                if val == best:
                    df.loc[idx, col] = f"\\textbf{{{val:.2f}}}"
                elif second is not None and val == second:
                    df.loc[idx, col] = f"\\underline{{{val:.2f}}}"
                else:
                    df.loc[idx, col] = f"{val:.2f}"

    return df
mean_df = gcd_exp.groupby(["Metric", "Method"], as_index=False)[dataset_cols + ['Avg.']].mean().set_index(['Metric', 'Method']).round(2)
latex_df = highlight_best_second(mean_df, dataset_cols + ['Avg.'])
latex_code = latex_df.to_latex(
    escape=False,      # 必须，否则 \textbf 会被转义
    multirow=True,
    column_format="ll" + "c" * len(dataset_cols + ['Avg.'])
)

with open("latex_or_csv/gcd.tex", "w") as f:
    f.write(latex_code.replace(' ', ''))

/tmp/ipykernel_4055003/971230768.py:33: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  mean_df = gcd_exp.groupby(["Metric", "Method"], as_index=False)[dataset_cols + ['Avg.']].mean().set_index(['Metric', 'Method']).round(2)
/tmp/ipykernel_4055003/971230768.py:30: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '85.85' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, col] = f"{val:.2f}"
/tmp/ipykernel_4055003/971230768.py:30: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '62.69' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, col] = 